# 0. Ideas

```python
class VigenereCipher:
    def __init__(self, keyword: str):
        self.keyword = keyword.lower()
        self.alphabet = 'abcdefghijklmnopqrstuvwxyz'

    def shift_char(self, c, key_c, encode=True):
        if c not in self.alphabet:
            return c
        shift = self.alphabet.index(key_c)
        if not encode:
            shift = -shift
        return self.alphabet[(self.alphabet.index(c) + shift) % 26]

    def transform(self, text: str, encode=True):
        text = text.lower()
        result = []
        key_len = len(self.keyword)
        for i, c in enumerate(text):
            key_c = self.keyword[i % key_len]
            result.append(self.shift_char(c, key_c, encode))
        return ''.join(result)

    def encrypt(self, text: str) -> str:
        return self.transform(text, True)

    def decrypt(self, text: str) -> str:
        return self.transform(text, False)

if __name__ == "__main__":
    keyword = input("Enter encryption keyword: ").strip()
    cipher = VigenereCipher(keyword)

    plaintext = input("Enter text to encrypt: ").strip()
    encrypted = cipher.encrypt(plaintext)
    print(f"Encrypted text: {encrypted}")

    decrypted = cipher.decrypt(encrypted)
    print(f"Decrypted back: {decrypted}")
    ```
    

**Layering multiple encryption techniques** is a legitimate way to increase confidentiality, especially when combining reversible (symmetric/asymmetric ciphers) with non-reversible (hashing or key derivation) methods. Let’s break down a pipeline concept:

---

### 🔐 Step 1: Vigenère or Caesar (Lightweight Substitution)
A first-pass classical cipher adds basic scrambling and deters casual eyes. We’ve already built this.

---

### 🔁 Step 2: Symmetric Encryption (AES or ChaCha20)
Use a **modern symmetric cipher** like:
- **AES** (Advanced Encryption Standard) — widely adopted, secure, hardware-accelerated
- **ChaCha20** — faster in software and more resistant to timing attacks

🛠 You’d need a secure key and an Initialization Vector (IV) — both must be handled carefully.

---

### 🔑 Step 3: Asymmetric Encryption (RSA or ECC)
Wrap the **AES/ChaCha key using RSA** (public-key crypto). That way:
- Only the intended recipient (with the private key) can unwrap the symmetric key.
- This hybrid approach is how HTTPS and encrypted messaging apps work.

---

### 🔂 Step 4: Hashing + Signature (Optional but Powerful)
- **Hash (SHA-256)** the original text for **integrity checking**
- Sign the hash using a private key (e.g., **RSA signature** or **ECDSA**) to verify authenticity

---

### 🔁 Bonus: Polymorphic Layer
A **polymorphic cipher** randomly alters its encoding strategy per encryption session — changing keys, routes, or even algorithm subsets. It’s excellent for:
- Detecting tampering or replay
- Obfuscation-based defense (especially in malware counter-analysis)

---

### 📦 Example Pipeline Flow

```plaintext
[Plaintext]
   ↓ Vigenère (keyword obfuscation)
[Step 1 Output]
   ↓ AES (symmetric block encryption)
[Step 2 Output]
   ↓ RSA public key encryption (protect AES key)
[Encrypted + EncryptedKey]
   ↓ Hash + Sign (SHA-256 + RSA signature)
[Final Output Package]
```

---


We are venturing into thrilling territory: a **modular, secure cryptographic pipeline** with CLI orchestration. Think of it like building your own personal encryption factory. Here’s a blueprint you can iterate on, using Python’s `cryptography` library (or `PyCryptodome`, if preferred). In the following we lay out key stages and suggest how to wire them into a mini-pipeline or even a Dagster-lite setup.

---

### 🔐 Stage 1: Vigenère Cipher (Custom Classical Layer)

```python
def vigenere_encrypt(plaintext: str, key: str) -> str:
    from itertools import cycle
    alphabet = 'abcdefghijklmnopqrstuvwxyz'
    enc = []
    for c, k in zip(plaintext.lower(), cycle(key.lower())):
        if c in alphabet:
            shifted = (ord(c) - ord('a') + ord(k) - ord('a')) % 26
            enc.append(chr(ord('a') + shifted))
        else:
            enc.append(c)
    return ''.join(enc)
```

---

### 🔐 Stage 2: AES Encryption (from `cryptography`)

```python
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives import padding
from cryptography.hazmat.backends import default_backend
import os

def aes_encrypt(data: bytes, key: bytes) -> tuple[bytes, bytes, bytes]:
    iv = os.urandom(16)
    padder = padding.PKCS7(128).padder()
    padded_data = padder.update(data) + padder.finalize()
    cipher = Cipher(algorithms.AES(key), modes.CBC(iv), backend=default_backend())
    encryptor = cipher.encryptor()
    ct = encryptor.update(padded_data) + encryptor.finalize()
    return ct, iv, key
```

---

### 🔐 Stage 3: RSA Wrapper (Encrypting AES Key)

```python
from cryptography.hazmat.primitives.asymmetric import rsa, padding as asympadding
from cryptography.hazmat.primitives import serialization, hashes

def rsa_encrypt_key(aes_key: bytes, public_key) -> bytes:
    return public_key.encrypt(
        aes_key,
        asympadding.OAEP(
            mgf=asympadding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )
```

---

### 🔐 Stage 4: SHA-256 Hashing + Signing (Optional)

```python
from cryptography.hazmat.primitives.asymmetric import padding as asypad
from cryptography.hazmat.primitives import hashes

def sign_data(private_key, message: bytes) -> bytes:
    return private_key.sign(
        message,
        asypad.PSS(
            mgf=asypad.MGF1(hashes.SHA256()),
            salt_length=asypad.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )
```

---

### ⚙️ CLI Pipeline Script (Think: Entry Point)

```python
import argparse

def main():
    parser = argparse.ArgumentParser(description="Secure Encryption Pipeline")
    parser.add_argument("text", help="Text to encrypt")
    parser.add_argument("--vkey", required=True, help="Vigenère keyword")
    args = parser.parse_args()

    # Step 1: Vigenère
    v_encrypted = vigenere_encrypt(args.text, args.vkey)

    # Step 2: AES
    aes_key = os.urandom(32)
    aes_ct, iv, key = aes_encrypt(v_encrypted.encode(), aes_key)

    # Step 3: RSA
    # Load/generate RSA keys here
    # rsa_ct = rsa_encrypt_key(aes_key, public_key)

    print("Encrypted Text:", aes_ct.hex())

if __name__ == "__main__":
    main()
```

---

### 🚀 Orchestrating With a Free Dagster-Alternative

If Dagster is too heavy or too hosted, we may consider:
- **Prefect (open-source Core edition)** — beautiful CLI-first workflow engine
- **Airflow (via Docker)** — heavier but still usable locally
- **Plain `typer`/`argparse` + task runner** — lightweight and shell-friendly

Or even write a *custom* YAML + Python orchestrator that loads YAML pipeline definitions and runs each stage.


Here’s a **single code cell** version of a decryption pipeline that complements our encryption pipeline — reversing RSA → AES → Vigenère. It mimics our original style and format:

```python
def vigenere_decrypt(ciphertext: str, key: str) -> str:
    from itertools import cycle
    alphabet = 'abcdefghijklmnopqrstuvwxyz'
    dec = []
    for c, k in zip(ciphertext.lower(), cycle(key.lower())):
        if c in alphabet:
            shifted = (ord(c) - ord(k)) % 26
            dec.append(chr(ord('a') + shifted))
        else:
            dec.append(c)
    return ''.join(dec)

from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives import padding
from cryptography.hazmat.backends import default_backend

def aes_decrypt(ciphertext: bytes, key: bytes, iv: bytes) -> bytes:
    cipher = Cipher(algorithms.AES(key), modes.CBC(iv), backend=default_backend())
    decryptor = cipher.decryptor()
    padded_data = decryptor.update(ciphertext) + decryptor.finalize()
    unpadder = padding.PKCS7(128).unpadder()
    return unpadder.update(padded_data) + unpadder.finalize()

from cryptography.hazmat.primitives.asymmetric import padding as asympadding
from cryptography.hazmat.primitives import serialization, hashes

def rsa_decrypt_key(encrypted_key: bytes, private_key) -> bytes:
    return private_key.decrypt(
        encrypted_key,
        asympadding.OAEP(
            mgf=asympadding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )

import argparse

def main():
    parser = argparse.ArgumentParser(description="Secure Decryption Pipeline")
    parser.add_argument("ciphertext", help="AES ciphertext hex string")
    parser.add_argument("iv", help="Initialization vector (hex)")
    parser.add_argument("enc_key", help="RSA-encrypted AES key (hex)")
    parser.add_argument("priv_key_path", help="Path to PEM private key")
    parser.add_argument("--vkey", required=True, help="Vigenère keyword")
    args = parser.parse_args()

    # Load RSA Private Key
    with open(args.priv_key_path, "rb") as key_file:
        private_key = serialization.load_pem_private_key(
            key_file.read(),
            password=None,
            backend=default_backend()
        )

    # Step 1: RSA decrypt AES key
    encrypted_key = bytes.fromhex(args.enc_key)
    aes_key = rsa_decrypt_key(encrypted_key, private_key)

    # Step 2: AES decryption
    iv = bytes.fromhex(args.iv)
    aes_ct = bytes.fromhex(args.ciphertext)
    vigenere_out = aes_decrypt(aes_ct, aes_key, iv).decode()

    # Step 3: Vigenère decryption
    plain = vigenere_decrypt(vigenere_out, args.vkey)

    print("Decrypted Text:", plain)

if __name__ == "__main__":
    main()
```

It expects:
- the RSA-encrypted AES key (`--enc_key`)
- the AES ciphertext (`--ciphertext`) and IV (`--iv`) as hex
- the RSA private key file path
- the Vigenère keyword


Below is a **complete, reverse decryption pipeline** that unwinds every step of our original encryption process: RSA → AES → Vigenère. This version includes:

- RSA decryption of the AES key
- AES decryption of the Vigenère-encrypted content
- Vigenère decryption of the final plaintext
- CLI arguments and clear structure to match your encryption pipeline

We can paste this into a single Python file or cell:

```python
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives import padding, serialization, hashes
from cryptography.hazmat.primitives.asymmetric import padding as asympadding
from cryptography.hazmat.backends import default_backend
from itertools import cycle
import argparse

# Step 1: Vigenère Decryption
def vigenere_decrypt(ciphertext: str, key: str) -> str:
    alphabet = 'abcdefghijklmnopqrstuvwxyz'
    plaintext = []
    for c, k in zip(ciphertext.lower(), cycle(key.lower())):
        if c in alphabet:
            shift = (ord(c) - ord(k)) % 26
            plaintext.append(chr(ord('a') + shift))
        else:
            plaintext.append(c)
    return ''.join(plaintext)

# Step 2: AES Decryption
def aes_decrypt(ciphertext: bytes, key: bytes, iv: bytes) -> bytes:
    cipher = Cipher(algorithms.AES(key), modes.CBC(iv), backend=default_backend())
    decryptor = cipher.decryptor()
    padded_data = decryptor.update(ciphertext) + decryptor.finalize()
    unpadder = padding.PKCS7(128).unpadder()
    return unpadder.update(padded_data) + unpadder.finalize()

# Step 3: RSA Key Decryption
def rsa_decrypt_key(encrypted_key: bytes, private_key_path: str) -> bytes:
    with open(private_key_path, "rb") as key_file:
        private_key = serialization.load_pem_private_key(
            key_file.read(),
            password=None,
            backend=default_backend()
        )
    return private_key.decrypt(
        encrypted_key,
        asympadding.OAEP(
            mgf=asympadding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )

# Main CLI logic
def main():
    parser = argparse.ArgumentParser(description="Secure Decryption Pipeline")
    parser.add_argument("ciphertext", help="AES ciphertext as hex string")
    parser.add_argument("--iv", required=True, help="AES initialization vector (hex)")
    parser.add_argument("--rsa_key", required=True, help="Path to private RSA PEM key")
    parser.add_argument("--enc_key", required=True, help="RSA-encrypted AES key (hex)")
    parser.add_argument("--vkey", required=True, help="Vigenère keyword used at encryption")
    args = parser.parse_args()

    # Step 1: Decrypt AES key with RSA
    encrypted_key = bytes.fromhex(args.enc_key)
    aes_key = rsa_decrypt_key(encrypted_key, args.rsa_key)

    # Step 2: Decrypt AES-encrypted Vigenère text
    ciphertext = bytes.fromhex(args.ciphertext)
    iv = bytes.fromhex(args.iv)
    vigenere_encrypted = aes_decrypt(ciphertext, aes_key, iv).decode()

    # Step 3: Decrypt Vigenère to get plaintext
    original_text = vigenere_decrypt(vigenere_encrypted, args.vkey)

    print("\n✅ Decrypted Plaintext:", original_text)

if __name__ == "__main__":
    main()
```

---

### 🛠️ Sample Usage (Terminal):
```bash
python decrypt_pipeline.py \
  "62c99f..." \
  --iv "3f3ec2..." \
  --rsa_key private_key.pem \
  --enc_key "45ff19..." \
  --vkey lemon
```

# 1. Installation section

In [1]:
!pip install cryptography
!pip install pycryptodome
!pip install typer rich

In [3]:
!pip install dagster dagster-webserver
# Optional: for Jupyter support
!pip install dagster[jupyter]

# Optional: for filesystem and database IO management
!pip install dagster-io

# Optional: include example integrations
!pip install dagster pandas requests


ERROR: Could not find a version that satisfies the requirement dagster-io (from versions: none)
ERROR: No matching distribution found for dagster-io


# 2. Implementation section

## 2.1 Generate private and public keys

In [ ]:
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives import serialization
import os

# Ensure the rsa/ directory exists
os.makedirs("rsa", exist_ok=True)

# Generate private key
private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)

# Save private key
with open("root/rsa/private.pem", "wb") as f:
    f.write(
        private_key.private_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PrivateFormat.PKCS8,
            encryption_algorithm=serialization.NoEncryption()
        )
    )

# Save public key
public_key = private_key.public_key()
with open("root/rsa/public.pem", "wb") as f:
    f.write(
        public_key.public_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PublicFormat.SubjectPublicKeyInfo
        )
    )

## 2.2 Encryption pipeline

In [9]:
def vigenere_encrypt(plaintext: str, key: str) -> str:
    from itertools import cycle
    alphabet = 'abcdefghijklmnopqrstuvwxyz'
    enc = []
    for c, k in zip(plaintext.lower(), cycle(key.lower())):
        if c in alphabet:
            shifted = (ord(c) - ord('a') + ord(k) - ord('a')) % 26
            enc.append(chr(ord('a') + shifted))
        else:
            enc.append(c)
    return ''.join(enc)

from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives import padding
from cryptography.hazmat.backends import default_backend
import os

def aes_encrypt(data: bytes, key: bytes) -> tuple[bytes, bytes, bytes]:
    iv = os.urandom(16)
    padder = padding.PKCS7(128).padder()
    padded_data = padder.update(data) + padder.finalize()
    cipher = Cipher(algorithms.AES(key), modes.CBC(iv), backend=default_backend())
    encryptor = cipher.encryptor()
    ct = encryptor.update(padded_data) + encryptor.finalize()
    return ct, iv, key

from cryptography.hazmat.primitives.asymmetric import rsa, padding as asympadding
from cryptography.hazmat.primitives import serialization, hashes

def rsa_encrypt_key(aes_key: bytes, public_key) -> bytes:
    return public_key.encrypt(
        aes_key,
        asympadding.OAEP(
            mgf=asympadding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )

from cryptography.hazmat.primitives.asymmetric import padding as asypad
from cryptography.hazmat.primitives import hashes

def sign_data(private_key, message: bytes) -> bytes:
    return private_key.sign(
        message,
        asypad.PSS(
            mgf=asypad.MGF1(hashes.SHA256()),
            salt_length=asypad.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )

import argparse

def main():
    parser = argparse.ArgumentParser(description="Secure Encryption Pipeline")
    parser.add_argument("text", help="Text to encrypt")
    parser.add_argument("--vkey", required=True, help="Vigenère keyword")
    args = parser.parse_args()

    # Step 1: Vigenère
    v_encrypted = vigenere_encrypt(args.text, args.vkey)

    # Step 2: AES
    aes_key = os.urandom(32)
    aes_ct, iv, key = aes_encrypt(v_encrypted.encode(), aes_key)

    # Step 3: RSA
    # Load/generate RSA keys here
    # rsa_ct = rsa_encrypt_key(aes_key, public_key)

    print("Encrypted Text:", aes_ct.hex())

if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] --vkey VKEY text
ipykernel_launcher.py: error: the following arguments are required: --vkey


SystemExit: 2

C:\Users\balan\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3516: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 2.3 Example usage (bash)

Here’s an example of how one could use the encryption pipeline from a terminal, along with a demonstration of what happens at each step using mock data:

---

### ✅ Step 1: Save the Script

Save your pipeline code into a Python file, for example:

```
secure_pipeline.py
```

---

### ✅ Step 2: Run It From CLI

```bash
python secure_pipeline.py "This is top secret text." --vkey lemon
```

This triggers:

1. **Vigenère Encryption** of the plaintext using the keyword `lemon`  
   Output (example): `aohu wp hsd fvuvy vica.`

2. **AES Encryption** of the result (binary output encoded as hex)  
   Output (example): `afcb0192b8eaeffa1e3435a4da02c976f65411a7f71af5a56ce10833e6785021`

3. **RSA Encryption** (optional — in your code it’s currently commented out)

4. **Digital Signing** of the message (optional — defined but not yet called)

---

### 🔐 Example Output (simplified)

```plaintext
Encrypted Text: afcb0192b8eaeffa1e3435a4da02c976f65411a7f71af5a56ce10833e6785021
```

---

### 💡 Notes for Extension

- To make the pipeline fully functional, we should:
  - Load or generate a public/private RSA key pair
  - Implement `rsa_encrypt_key(aes_key, public_key)` and print/store the result
  - Store or transmit the IV and the RSA-encrypted AES key securely
  - Optionally sign the AES ciphertext with `sign_data(...)`

## 2.4 Decryption pipeline

In [11]:
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives import padding, serialization, hashes
from cryptography.hazmat.primitives.asymmetric import padding as asympadding
from cryptography.hazmat.backends import default_backend
from itertools import cycle
import argparse

# Step 1: Vigenère Decryption
def vigenere_decrypt(ciphertext: str, key: str) -> str:
    alphabet = 'abcdefghijklmnopqrstuvwxyz'
    plaintext = []
    for c, k in zip(ciphertext.lower(), cycle(key.lower())):
        if c in alphabet:
            shift = (ord(c) - ord(k)) % 26
            plaintext.append(chr(ord('a') + shift))
        else:
            plaintext.append(c)
    return ''.join(plaintext)

# Step 2: AES Decryption
def aes_decrypt(ciphertext: bytes, key: bytes, iv: bytes) -> bytes:
    cipher = Cipher(algorithms.AES(key), modes.CBC(iv), backend=default_backend())
    decryptor = cipher.decryptor()
    padded_data = decryptor.update(ciphertext) + decryptor.finalize()
    unpadder = padding.PKCS7(128).unpadder()
    return unpadder.update(padded_data) + unpadder.finalize()

# Step 3: RSA Key Decryption
def rsa_decrypt_key(encrypted_key: bytes, private_key_path: str) -> bytes:
    with open(private_key_path, "rb") as key_file:
        private_key = serialization.load_pem_private_key(
            key_file.read(),
            password=None,
            backend=default_backend()
        )
    return private_key.decrypt(
        encrypted_key,
        asympadding.OAEP(
            mgf=asympadding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )

# Main CLI logic
def main():
    parser = argparse.ArgumentParser(description="Secure Decryption Pipeline")
    parser.add_argument("ciphertext", help="AES ciphertext as hex string")
    parser.add_argument("--iv", required=True, help="AES initialization vector (hex)")
    parser.add_argument("--rsa_key", required=True, help="Path to private RSA PEM key")
    parser.add_argument("--enc_key", required=True, help="RSA-encrypted AES key (hex)")
    parser.add_argument("--vkey", required=True, help="Vigenère keyword used at encryption")
    args = parser.parse_args()

    # Step 1: Decrypt AES key with RSA
    encrypted_key = bytes.fromhex(args.enc_key)
    aes_key = rsa_decrypt_key(encrypted_key, args.rsa_key)

    # Step 2: Decrypt AES-encrypted Vigenère text
    ciphertext = bytes.fromhex(args.ciphertext)
    iv = bytes.fromhex(args.iv)
    vigenere_encrypted = aes_decrypt(ciphertext, aes_key, iv).decode()

    # Step 3: Decrypt Vigenère to get plaintext
    original_text = vigenere_decrypt(vigenere_encrypted, args.vkey)

    print("\n✅ Decrypted Plaintext:", original_text)

if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] --iv IV --rsa_key RSA_KEY --enc_key ENC_KEY
                             --vkey VKEY
                             ciphertext
ipykernel_launcher.py: error: the following arguments are required: --iv, --rsa_key, --enc_key, --vkey


SystemExit: 2

## 2.5 Example usage (bash)

```bash
python decrypt_pipeline.py \
  "62c99f..." \
  --iv "3f3ec2..." \
  --rsa_key private_key.pem \
  --enc_key "45ff19..." \
  --vkey lemon
```

# 3. Dagster execution

We are ready to bring in Dagster to orchestrate your encryption and decryption flows, here's how one can wire it all together. We’ll treat `secure_pipeline.py` and `decrypt_pipeline.py` as modular components and then use Dagster to manage the flow, parameters, and runtime configuration.

---

### 📦 Step 1: Project Structure

```
project_root/
│
├── pipelines/
│   ├── secure_pipeline.py        # contains encrypt logic
│   ├── decrypt_pipeline.py       # contains decrypt logic
│
├── dagster_pipelines/
│   ├── __init__.py
│   ├── jobs.py                   # Dagster job definitions
│   └── ops.py                    # Dagster steps
│
├── workspace.yaml
└── dagster.yaml
```

---

### ⚙️ Step 2: Define Dagster Ops (ops.py)

```python
from dagster import op
from pipelines import secure_pipeline, decrypt_pipeline

@op
def encrypt_op(context, text: str, vkey: str):
    return secure_pipeline.encrypt_main(text, vkey)  # create `encrypt_main()` in secure_pipeline.py

@op
def decrypt_op(context, encrypted_data: dict, vkey: str, rsa_key_path: str):
    return decrypt_pipeline.decrypt_main(
        ciphertext=encrypted_data['ciphertext'],
        iv=encrypted_data['iv'],
        enc_key=encrypted_data['enc_key'],
        priv_key_path=rsa_key_path,
        vkey=vkey
    )
```

---

### 🔧 Step 3: Define the Dagster Job (jobs.py)

```python
from dagster import job
from .ops import encrypt_op, decrypt_op

@job
def encryption_decryption_job():
    decrypted = decrypt_op(
        encrypted_data=encrypt_op(),
        vkey="lemon",
        rsa_key_path="rsa_keys/private_key.pem"
    )
```

We can also parameterize the job with `@configurable` or use Dagster’s config schema if one wants to expose CLI/graph inputs.

---

### 🔌 Step 4: Entry Point (CLI or UI)

From the terminal:

```bash
dagster dev   # launches Dagster UI
```

Or from a Python CLI:

```bash
from dagster_pipelines.jobs import encryption_decryption_job
encryption_decryption_job.execute_in_process()
```

---

### 🧩 How our secure_pipeline.py and decrypt_pipeline.py Should Look

Each should expose a callable like this:

```python
# secure_pipeline.py
def encrypt_main(text: str, vkey: str) -> dict:
    # return {
    #   "ciphertext": <hex>,
    #   "iv": <hex>,
    #   "enc_key": <hex>
    # }
```

```python
# decrypt_pipeline.py
def decrypt_main(ciphertext, iv, enc_key, priv_key_path, vkey) -> str:
    # return plaintext
```

---

### 🚀 Ready to Run

This turns our CLI tools into reusable data pipeline steps. Add logging, tests, or run schedules and we are on our way to production-grade cryptography orchestration.

To get up and running with **Dagster**, we will want to install its core package along with any extras you plan to use (like AWS, dbt, Airbyte, etc). But for a basic local development setup with the web UI, here’s what we need:

```bash
!pip install dagster dagster-webserver
```

This gives us:
- `dagster`: core library for defining jobs, ops, resources, etc.
- `dagster-webserver`: the lightweight UI you can launch with `dagster dev`

If we are using extras like Jupyter, SQLAlchemy, or cloud integrations, you can layer in specific extras:

```bash
# Optional: for Jupyter support
!pip install dagster[jupyter]

# Optional: for filesystem and database IO management
!pip install dagster-io

# Optional: include example integrations
!pip install dagster pandas requests
```

Once installed, we launch our dev server like this in a terminal:

```bash
dagster dev
```

And our UI will be live at [http://localhost:3000](http://localhost:3000) 🎛️

## 3.1 Generate private and public keys

In [5]:
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives import serialization
import os

# Ensure the rsa/ directory exists
os.makedirs("rsa", exist_ok=True)

# Generate private key
private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)

# Save private key
with open("root/rsa/private.pem", "wb") as f:
    f.write(
        private_key.private_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PrivateFormat.PKCS8,
            encryption_algorithm=serialization.NoEncryption()
        )
    )

# Save public key
public_key = private_key.public_key()
with open("root/rsa/public.pem", "wb") as f:
    f.write(
        public_key.public_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PublicFormat.SubjectPublicKeyInfo
        )
    )

In [5]:
import os
print(os.getcwd())

C:\Users\balan\OneDrive\Desktop\EncryptionPipeline


## 3.2 Execute the dagster pipeline

In [ ]:
import os
os.chdir("C:\\Users\\balan\\OneDrive\\Desktop\\EncryptionPipeline\\root")
!dagster dev -w workspace.yaml